In [6]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import matplotlib
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error

# 设置matplotlib中文字体
matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei']
matplotlib.rcParams['axes.unicode_minus'] = False

# 第一部分：数据预处理
print("第一部分：数据预处理")

# 1.1 房价数据预处理
print("\n1.1 房价数据预处理")

train_price = pd.read_csv('train_price.csv')
test_price = pd.read_csv('test_price.csv')
print(f"原始数据 - 训练集: {train_price.shape}, 测试集: {test_price.shape}")

def preprocess_price(df):
    df = df.copy()

    df['环线'] = df['环线'].fillna('未知')
    df['房屋户型'] = df['房屋户型'].fillna(df['房屋户型'].mode()[0])

    df['建筑面积'] = df['建筑面积'].apply(lambda x: float(re.findall(r'\d+\.?\d*', str(x))[0]) if pd.notna(x) and re.findall(r'\d+\.?\d*', str(x)) else np.nan)
    df['套内面积'] = df['套内面积'].apply(lambda x: float(re.findall(r'\d+\.?\d*', str(x))[0]) if pd.notna(x) and re.findall(r'\d+\.?\d*', str(x)) else np.nan)

    train_mask = df['套内面积'].notna() & df['建筑面积'].notna()
    if train_mask.sum() > 0:
        lr = LinearRegression()
        lr.fit(df.loc[train_mask, ['建筑面积']], df.loc[train_mask, '套内面积'])
        pred_mask = df['套内面积'].isna() & df['建筑面积'].notna()
        df.loc[pred_mask, '套内面积'] = lr.predict(df.loc[pred_mask, ['建筑面积']])

    df['房屋朝向'] = df['房屋朝向'].fillna(df['房屋朝向'].mode()[0])
    df['建筑结构'] = df['建筑结构'].fillna(df['建筑结构'].mode()[0])
    df['装修情况'] = df['装修情况'].fillna(df['装修情况'].mode()[0])

    def parse_layout(x):
        if pd.isna(x):
            return 0, 0, 0, 0
        x = str(x)
        室 = re.findall(r'(\d+)室', x)
        厅 = re.findall(r'(\d+)厅', x)
        厨 = re.findall(r'(\d+)厨', x)
        卫 = re.findall(r'(\d+)卫', x)
        房间 = re.findall(r'(\d+)房间', x)
        室数 = int(室[0]) if 室 else (int(房间[0]) if 房间 else 0)
        厅数 = int(厅[0]) if 厅 else 0
        厨数 = int(厨[0]) if 厨 else 0
        卫数 = int(卫[0]) if 卫 else 0
        return 室数, 厅数, 厨数, 卫数

    df[['室数', '厅数', '厨数', '卫数']] = df['房屋户型'].apply(lambda x: pd.Series(parse_layout(x)))
    df = df.drop('房屋户型', axis=1)

    def parse_floor(x):
        if pd.isna(x):
            return np.nan
        x = str(x)
        floor_types = ['地下室', '底层', '低楼层', '中楼层', '高楼层', '顶层']
        for ft in floor_types:
            if ft in x:
                return ft
        return np.nan

    df['所在楼层'] = df['所在楼层'].apply(parse_floor)
    df['所在楼层'] = df['所在楼层'].fillna(df['所在楼层'].mode()[0] if len(df['所在楼层'].mode()) > 0 else '中楼层')

    df['梯户比例'] = df['梯户比例'].fillna(df['梯户比例'].mode()[0])
    df['配备电梯'] = df['配备电梯'].fillna(df['配备电梯'].mode()[0])
    df = df.drop(['别墅类型', '上次交易', '抵押信息'], axis=1)

    df['房屋用途'] = df['房屋用途'].fillna(df['房屋用途'].mode()[0])
    df['房屋年限'] = df['房屋年限'].fillna(df['房屋年限'].mode()[0])

    df['是否近地铁'] = df['房屋优势'].fillna('').str.contains('地铁', na=False).astype(int)
    df['是否满五年'] = df['房屋优势'].fillna('').str.contains('满五', na=False).astype(int)
    df['是否装修'] = df['房屋优势'].fillna('').str.contains('装修', na=False).astype(int)
    df = df.drop('房屋优势', axis=1)

    df = df.drop(['核心卖点', '户型介绍'], axis=1)

    df['是否有医院'] = df['周边配套'].fillna('').str.contains('医院', na=False).astype(int)
    df['是否有超市'] = df['周边配套'].fillna('').str.contains('超市', na=False).astype(int)
    df['是否有公园'] = df['周边配套'].fillna('').str.contains('公园', na=False).astype(int)
    df['是否有幼儿园'] = df['周边配套'].fillna('').str.contains('幼儿园', na=False).astype(int)
    df['是否有银行'] = df['周边配套'].fillna('').str.contains('银行', na=False).astype(int)
    df = df.drop('周边配套', axis=1)

    df['是否有公交'] = df['交通出行'].fillna('').str.contains('公交', na=False).astype(int)
    df = df.drop('交通出行', axis=1)

    df['区县'] = df['区县'].fillna(0)
    df['板块_comm'] = df['板块_comm'].fillna(0)

    df['环线位置'] = df['环线位置'].fillna('未知')
    df['物业类别'] = df['物业类别'].fillna('未知')

    df['是否平房_物业'] = df['物业类别'].fillna('').str.contains('平房', na=False).astype(int)
    df['是否普通住宅'] = df['物业类别'].fillna('').str.contains('普通住宅', na=False).astype(int)
    df['是否车库'] = df['物业类别'].fillna('').str.contains('车库', na=False).astype(int)
    df['是否底商'] = df['物业类别'].fillna('').str.contains('底商', na=False).astype(int)
    df = df.drop('物业类别', axis=1)

    def parse_year(x):
        if pd.isna(x):
            return np.nan, np.nan
        years = re.findall(r'\d{4}', str(x))
        if len(years) == 0:
            return np.nan, np.nan
        elif len(years) == 1:
            return float(years[0]), float(years[0])
        else:
            return float(years[0]), float(years[-1])

    df[['开工时间', '完工时间']] = df['建筑年代'].apply(lambda x: pd.Series(parse_year(x)))
    start_median = df['开工时间'].median()
    end_median = df['完工时间'].median()
    if end_median > start_median:
        df['开工时间'] = df['开工时间'].fillna(start_median)
        df['完工时间'] = df['完工时间'].fillna(end_median)
    else:
        df['开工时间'] = df['开工时间'].fillna(end_median)
        df['完工时间'] = df['完工时间'].fillna(end_median)
    df = df.drop('建筑年代', axis=1)

    df['开发商'] = df['开发商'].fillna('未知')

    df['房屋总数'] = df['房屋总数'].apply(lambda x: float(re.findall(r'\d+', str(x))[0]) if pd.notna(x) and re.findall(r'\d+', str(x)) else np.nan)
    df['房屋总数'] = df['房屋总数'].fillna(df['房屋总数'].median())
    df['楼栋总数'] = df['楼栋总数'].apply(lambda x: float(re.findall(r'\d+', str(x))[0]) if pd.notna(x) and re.findall(r'\d+', str(x)) else np.nan)
    df['楼栋总数'] = df['楼栋总数'].fillna(df['楼栋总数'].median())

    df['物业公司'] = df['物业公司'].fillna('未知')

    df['绿 化 率'] = df['绿 化 率'].apply(lambda x: float(re.findall(r'\d+\.?\d*', str(x))[0]) if pd.notna(x) and re.findall(r'\d+\.?\d*', str(x)) else np.nan)
    df['绿 化 率'] = df['绿 化 率'].fillna(df['绿 化 率'].median())
    df['容 积 率'] = df['容 积 率'].fillna(df['容 积 率'].median())

    def parse_fee(x):
        if pd.isna(x):
            return np.nan
        nums = re.findall(r'\d+\.?\d*', str(x))
        if len(nums) == 0:
            return np.nan
        return np.mean([float(n) for n in nums])

    df['物 业 费'] = df['物 业 费'].apply(parse_fee)
    df['物 业 费'] = df['物 业 费'].fillna(df['物 业 费'].median())

    df['是否塔板结合'] = df['建筑结构_comm'].fillna('').str.contains('塔板结合', na=False).astype(int)
    df['是否板楼'] = df['建筑结构_comm'].fillna('').str.contains('板楼', na=False).astype(int)
    df['是否平房_建筑'] = df['建筑结构_comm'].fillna('').str.contains('平房', na=False).astype(int)
    df = df.drop('建筑结构_comm', axis=1)

    df = df.drop('物业办公电话', axis=1)

    df['是否商品房'] = df['产权描述'].fillna('').str.contains('商品房', na=False).astype(int)
    df['是否限价商品房'] = df['产权描述'].fillna('').str.contains('限价', na=False).astype(int)
    df['是否已购公房'] = df['产权描述'].fillna('').str.contains('已购公房', na=False).astype(int)
    df['是否私产'] = df['产权描述'].fillna('').str.contains('私产', na=False).astype(int)
    df['是否央产房'] = df['产权描述'].fillna('').str.contains('央产', na=False).astype(int)
    df = df.drop('产权描述', axis=1)

    df['供水'] = df['供水'].fillna(df['供水'].mode()[0] if len(df['供水'].mode()) > 0 else '未知')
    df['供暖'] = df['供暖'].fillna(df['供暖'].mode()[0] if len(df['供暖'].mode()) > 0 else '未知')
    df['供电'] = df['供电'].fillna(df['供电'].mode()[0] if len(df['供电'].mode()) > 0 else '未知')

    df['燃气费'] = df['燃气费'].apply(parse_fee)
    df['燃气费'] = df['燃气费'].fillna(df['燃气费'].median())
    df['供热费'] = df['供热费'].apply(parse_fee)
    df['供热费'] = df['供热费'].fillna(df['供热费'].median())

    df['停车位'] = df['停车位'].fillna(df['停车位'].median())

    def parse_parking_fee(x):
        if pd.isna(x) or str(x) == '暂无':
            return 0
        nums = re.findall(r'\d+\.?\d*', str(x))
        if len(nums) == 0:
            return 0
        return np.mean([float(n) for n in nums])

    df['停车费用'] = df['停车费用'].apply(parse_parking_fee)

    cols_to_drop = ['coord_x', 'coord_y', '客户反馈']
    df = df.drop([col for col in cols_to_drop if col in df.columns], axis=1)

    return df

train_price_processed = preprocess_price(train_price)
test_price_processed = preprocess_price(test_price)
print(f"预处理后 - 训练集: {train_price_processed.shape}, 测试集: {test_price_processed.shape}")

# 1.2 租金数据预处理
print("\n1.2 租金数据预处理")

train_rent = pd.read_csv('train_rent.csv')
test_rent = pd.read_csv('test_rent.csv')
print(f"原始数据 - 训练集: {train_rent.shape}, 测试集: {test_rent.shape}")

def preprocess_rent(df):
    df = df.copy()

    df['户型'] = df['户型'].fillna(df['户型'].mode()[0] if len(df['户型'].mode()) > 0 else '未知')
    df['装修'] = df['装修'].fillna('未知')

    def parse_floor_level(x):
        if pd.isna(x):
            return np.nan
        x = str(x)
        if '低楼层' in x:
            return '低楼层'
        elif '中楼层' in x:
            return '中楼层'
        elif '高楼层' in x:
            return '高楼层'
        match = re.search(r'(\d+)/(\d+)', x)
        if match:
            current_floor = int(match.group(1))
            total_floor = int(match.group(2))
            if total_floor > 0:
                ratio = current_floor / total_floor
                if ratio <= 0.33:
                    return '低楼层'
                elif ratio <= 0.67:
                    return '中楼层'
                else:
                    return '高楼层'
        return np.nan

    df['楼层'] = df['楼层'].apply(parse_floor_level)
    df['楼层'] = df['楼层'].fillna(df['楼层'].mode()[0] if len(df['楼层'].mode()) > 0 else '中楼层')

    df['付款方式'] = df['付款方式'].apply(lambda x: np.nan if pd.notna(x) and 'https://image1.ljcdn.com/rent-' in str(x) else x)
    df['付款方式'] = df['付款方式'].fillna('暂未确定')

    df['电梯'] = df['电梯'].fillna(df['电梯'].mode()[0] if len(df['电梯'].mode()) > 0 else '无')
    df['车位'] = df['车位'].fillna('免费使用')

    df['用水'] = df['用水'].fillna(df['用水'].mode()[0] if len(df['用水'].mode()) > 0 else '民用水')
    df['用电'] = df['用电'].fillna(df['用电'].mode()[0] if len(df['用电'].mode()) > 0 else '民用电')
    df['燃气'] = df['燃气'].fillna(df['燃气'].mode()[0] if len(df['燃气'].mode()) > 0 else '天然气')
    df['采暖'] = df['采暖'].fillna(df['采暖'].mode()[0] if len(df['采暖'].mode()) > 0 else '集中供暖')

    df['租期'] = df['租期'].fillna('可以商量')

    df['有洗衣机'] = df['配套设施'].fillna('').str.contains('洗衣机', na=False).astype(int)
    df['有空调'] = df['配套设施'].fillna('').str.contains('空调', na=False).astype(int)
    df['有电视机'] = df['配套设施'].fillna('').str.contains('电视', na=False).astype(int)
    df['有冰箱'] = df['配套设施'].fillna('').str.contains('冰箱', na=False).astype(int)
    df['有热水器'] = df['配套设施'].fillna('').str.contains('热水器', na=False).astype(int)
    df['有天然气'] = df['配套设施'].fillna('').str.contains('天然气', na=False).astype(int)
    df = df.drop('配套设施', axis=1)

    df['区县'] = df['区县'].fillna(0)
    df['板块'] = df['板块'].fillna(0)

    cols_to_drop = ['coord_x', 'coord_y', '客户反馈']
    df = df.drop([col for col in cols_to_drop if col in df.columns], axis=1)

    def parse_parking_fee(x):
        if pd.isna(x) or str(x) == '暂无':
            return 0
        x = str(x)
        nums = re.findall(r'\d+\.?\d*', x)
        if len(nums) == 0:
            return 0
        nums = [float(n) for n in nums]
        return np.mean(nums)

    df['停车费用'] = df['停车费用'].apply(parse_parking_fee)
    df['停车位'] = df['停车位'].fillna(df['停车位'].median())

    if '物业办公电话' in df.columns:
        df = df.drop('物业办公电话', axis=1)

    def parse_utility_fee(x):
        if pd.isna(x):
            return np.nan
        x = str(x)
        nums = re.findall(r'\d+\.?\d*', x)
        if len(nums) == 0:
            return np.nan
        return np.mean([float(n) for n in nums])

    df['燃气费'] = df['燃气费'].apply(parse_utility_fee)
    df['燃气费'] = df['燃气费'].fillna(df['燃气费'].median())
    df['供热费'] = df['供热费'].apply(parse_utility_fee)
    df['供热费'] = df['供热费'].fillna(df['供热费'].median())

    df['绿 化 率'] = df['绿 化 率'].apply(lambda x: float(re.findall(r'\d+\.?\d*', str(x))[0]) if pd.notna(x) and re.findall(r'\d+\.?\d*', str(x)) else np.nan)
    df['绿 化 率'] = df['绿 化 率'].fillna(df['绿 化 率'].median())

    df['容 积 率'] = df['容 积 率'].fillna(df['容 积 率'].median())

    df['物 业 费'] = df['物 业 费'].apply(parse_utility_fee)
    df['物 业 费'] = df['物 业 费'].fillna(df['物 业 费'].median())

    df['是否塔板结合'] = df['建筑结构'].fillna('').str.contains('塔板结合', na=False).astype(int)
    df['是否板楼'] = df['建筑结构'].fillna('').str.contains('板楼', na=False).astype(int)
    df['是否平房_建筑'] = df['建筑结构'].fillna('').str.contains('平房', na=False).astype(int)
    df = df.drop('建筑结构', axis=1)

    df['供水'] = df['供水'].fillna(df['供水'].mode()[0] if len(df['供水'].mode()) > 0 else '未知')
    df['供暖'] = df['供暖'].fillna(df['供暖'].mode()[0] if len(df['供暖'].mode()) > 0 else '未知')
    df['供电'] = df['供电'].fillna(df['供电'].mode()[0] if len(df['供电'].mode()) > 0 else '未知')

    df['朝向'] = df['朝向'].fillna(df['朝向'].mode()[0] if len(df['朝向'].mode()) > 0 else '南')

    def parse_transaction_time(x):
        if pd.isna(x):
            return np.nan, np.nan
        x = str(x)
        match = re.search(r'(\d{4})-(\d{1,2})', x)
        if match:
            year = int(match.group(1))
            month = int(match.group(2))
            return year, month
        return np.nan, np.nan

    df[['交易年份', '交易月份']] = df['交易时间'].apply(lambda x: pd.Series(parse_transaction_time(x)))
    df['交易年份'] = df['交易年份'].fillna(df['交易年份'].median())
    df['交易月份'] = df['交易月份'].fillna(df['交易月份'].median())
    df = df.drop('交易时间', axis=1)

    df['是否平房_物业'] = df['物业类别'].fillna('').str.contains('平房', na=False).astype(int)
    df['是否普通住宅'] = df['物业类别'].fillna('').str.contains('普通住宅', na=False).astype(int)
    df['是否车库'] = df['物业类别'].fillna('').str.contains('车库', na=False).astype(int)
    df['是否底商'] = df['物业类别'].fillna('').str.contains('底商', na=False).astype(int)
    df = df.drop('物业类别', axis=1)

    def parse_year(x):
        if pd.isna(x):
            return np.nan, np.nan
        years = re.findall(r'\d{4}', str(x))
        if len(years) == 0:
            return np.nan, np.nan
        elif len(years) == 1:
            return float(years[0]), float(years[0])
        else:
            return float(years[0]), float(years[-1])

    df[['开工时间', '完工时间']] = df['建筑年代'].apply(lambda x: pd.Series(parse_year(x)))
    start_median = df['开工时间'].median()
    end_median = df['完工时间'].median()
    if end_median > start_median:
        df['开工时间'] = df['开工时间'].fillna(start_median)
        df['完工时间'] = df['完工时间'].fillna(end_median)
    else:
        df['开工时间'] = df['开工时间'].fillna(end_median)
        df['完工时间'] = df['完工时间'].fillna(end_median)
    df = df.drop('建筑年代', axis=1)

    if '环线位置' in df.columns:
        df['环线位置'] = df['环线位置'].fillna('未知')

    if '开发商' in df.columns:
        df['开发商'] = df['开发商'].fillna('未知')

    if '房屋总数' in df.columns:
        df['房屋总数'] = df['房屋总数'].apply(lambda x: float(re.findall(r'\d+', str(x))[0]) if pd.notna(x) and re.findall(r'\d+', str(x)) else np.nan)
        df['房屋总数'] = df['房屋总数'].fillna(df['房屋总数'].median())

    if '楼栋总数' in df.columns:
        df['楼栋总数'] = df['楼栋总数'].apply(lambda x: float(re.findall(r'\d+', str(x))[0]) if pd.notna(x) and re.findall(r'\d+', str(x)) else np.nan)
        df['楼栋总数'] = df['楼栋总数'].fillna(df['楼栋总数'].median())

    if '物业公司' in df.columns:
        df['物业公司'] = df['物业公司'].fillna('未知')

    df['是否商品房'] = df['产权描述'].fillna('').str.contains('商品房', na=False).astype(int)
    df['是否限价商品房'] = df['产权描述'].fillna('').str.contains('限价', na=False).astype(int)
    df['是否已购公房'] = df['产权描述'].fillna('').str.contains('已购公房', na=False).astype(int)
    df['是否私产'] = df['产权描述'].fillna('').str.contains('私产', na=False).astype(int)
    df['是否央产房'] = df['产权描述'].fillna('').str.contains('央产', na=False).astype(int)
    df = df.drop('产权描述', axis=1)

    df['面积'] = df['面积'].apply(lambda x: float(re.findall(r'\d+\.?\d*', str(x))[0]) if pd.notna(x) and re.findall(r'\d+\.?\d*', str(x)) else np.nan)
    df['面积'] = df['面积'].fillna(df['面积'].median())

    def parse_layout(x):
        if pd.isna(x):
            return 0, 0, 0, 0
        x = str(x)
        室 = re.findall(r'(\d+)室', x)
        厅 = re.findall(r'(\d+)厅', x)
        厨 = re.findall(r'(\d+)厨', x)
        卫 = re.findall(r'(\d+)卫', x)
        房间 = re.findall(r'(\d+)房间', x)
        室数 = int(室[0]) if 室 else (int(房间[0]) if 房间 else 0)
        厅数 = int(厅[0]) if 厅 else 0
        厨数 = int(厨[0]) if 厨 else 0
        卫数 = int(卫[0]) if 卫 else 0
        return 室数, 厅数, 厨数, 卫数

    df[['室数', '厅数', '厨数', '卫数']] = df['户型'].apply(lambda x: pd.Series(parse_layout(x)))
    df = df.drop('户型', axis=1)

    return df

train_rent_processed = preprocess_rent(train_rent)
test_rent_processed = preprocess_rent(test_rent)
print(f"预处理后 - 训练集: {train_rent_processed.shape}, 测试集: {test_rent_processed.shape}")


第一部分：数据预处理

1.1 房价数据预处理


/var/folders/xl/frpg_84d2ys1y_w6_xp1zz_c0000gn/T/ipykernel_4772/2057707652.py:21: DtypeWarning: Columns (3,32,34,43,46,49,51) have mixed types. Specify dtype option on import or set low_memory=False.
  train_price = pd.read_csv('train_price.csv')
/var/folders/xl/frpg_84d2ys1y_w6_xp1zz_c0000gn/T/ipykernel_4772/2057707652.py:22: DtypeWarning: Columns (4,32) have mixed types. Specify dtype option on import or set low_memory=False.
  test_price = pd.read_csv('test_price.csv')


原始数据 - 训练集: (103871, 55), 测试集: (34017, 55)
预处理后 - 训练集: (103871, 65), 测试集: (34017, 65)

1.2 租金数据预处理


/var/folders/xl/frpg_84d2ys1y_w6_xp1zz_c0000gn/T/ipykernel_4772/2057707652.py:205: DtypeWarning: Columns (23) have mixed types. Specify dtype option on import or set low_memory=False.
  train_rent = pd.read_csv('train_rent.csv')


原始数据 - 训练集: (98899, 46), 测试集: (9773, 46)
预处理后 - 训练集: (98899, 61), 测试集: (9773, 61)


In [7]:

# 第二部分：特征工程
print("\n第二部分：特征工程")

# 2.1 房价特征工程
print("\n2.1 房价特征工程")

def engineer_price_features(df):
    df = df.copy()

    df['卫浴配比'] = df['卫数'] / (df['室数'] + 1)
    df['卫浴配比'] = df['卫浴配比'].fillna(0)

    df['套内使用率'] = df['套内面积'] / df['建筑面积']
    df['套内使用率'] = df['套内使用率'].fillna(df['套内使用率'].median())

    df['周边配套得分'] = (df['是否有医院'] + df['是否有超市'] + df['是否有公园'] +
                          df['是否有幼儿园'] + df['是否有银行'])

    df['房龄'] = 2025 - df['完工时间']
    df['房龄'] = df['房龄'].clip(lower=0)

    df['小区品质指标'] = (1 / (df['容 积 率'] + 0.1)) * 10 + df['绿 化 率'] / 10
    df['小区品质指标'] = df['小区品质指标'].fillna(df['小区品质指标'].median())

    log_features = ['建筑面积', '套内面积', '房屋总数', '楼栋总数', '停车位']
    for feat in log_features:
        if feat in df.columns:
            df[f'{feat}_log'] = np.log1p(df[feat])

    df['面积_房龄交互'] = df['建筑面积'] * df['房龄']
    df['室数_面积交互'] = df['室数'] * df['建筑面积']
    df['容积率_绿化率交互'] = df['容 积 率'] * df['绿 化 率']
    df['配套_地铁交互'] = df['周边配套得分'] * df['是否近地铁']

    df['建筑面积_平方'] = df['建筑面积'] ** 2
    df['房龄_平方'] = df['房龄'] ** 2
    df['套内使用率_平方'] = df['套内使用率'] ** 2

    return df

train_price_processed = engineer_price_features(train_price_processed)
test_price_processed = engineer_price_features(test_price_processed)
print(f"特征工程后 - 训练集: {train_price_processed.shape}, 测试集: {test_price_processed.shape}")

price = train_price_processed['Price']
mean = price.mean()
std = price.std()
z_scores = np.abs((price - mean) / std)
outliers_mask = z_scores > 3
outliers_count = outliers_mask.sum()
print(f"房价异常值数量: {outliers_count} ({outliers_count/len(price)*100:.2f}%)")

train_price_processed = train_price_processed[~outliers_mask].reset_index(drop=True)
print(f"去除异常值后 - 训练集: {train_price_processed.shape}")

# 2.2 租金特征工程
print("\n2.2 租金特征工程")

def engineer_rent_features(df):
    df = df.copy()

    df['卫浴配比'] = df['卫数'] / (df['室数'] + 1)
    df['卫浴配比'] = df['卫浴配比'].fillna(0)

    df['房龄'] = 2025 - df['完工时间']
    df['房龄'] = df['房龄'].clip(lower=0)

    df['小区品质指标'] = (1 / (df['容 积 率'] + 0.1)) * 10 + df['绿 化 率'] / 10
    df['小区品质指标'] = df['小区品质指标'].fillna(df['小区品质指标'].median())

    df['配套设施得分'] = (df['有洗衣机'] + df['有空调'] + df['有电视机'] +
                          df['有冰箱'] + df['有热水器'] + df['有天然气'])

    df['交易时间趋势'] = df['交易年份'] * 12 + df['交易月份']

    log_features = ['面积', '房屋总数', '楼栋总数', '停车位']
    for feat in log_features:
        if feat in df.columns:
            df[f'{feat}_log'] = np.log1p(df[feat])

    df['面积_房龄交互'] = df['面积'] * df['房龄']
    df['室数_面积交互'] = df['室数'] * df['面积']
    df['容积率_绿化率交互'] = df['容 积 率'] * df['绿 化 率']
    df['配套_时间交互'] = df['配套设施得分'] * df['交易时间趋势']

    df['面积_平方'] = df['面积'] ** 2
    df['房龄_平方'] = df['房龄'] ** 2
    df['配套设施得分_平方'] = df['配套设施得分'] ** 2

    return df

train_rent_processed = engineer_rent_features(train_rent_processed)
test_rent_processed = engineer_rent_features(test_rent_processed)
print(f"特征工程后 - 训练集: {train_rent_processed.shape}, 测试集: {test_rent_processed.shape}")

price_rent = train_rent_processed['Price']
mean_rent = price_rent.mean()
std_rent = price_rent.std()
z_scores_rent = np.abs((price_rent - mean_rent) / std_rent)
outliers_mask_rent = z_scores_rent > 3
outliers_count_rent = outliers_mask_rent.sum()
print(f"租金异常值数量: {outliers_count_rent} ({outliers_count_rent/len(price_rent)*100:.2f}%)")

train_rent_processed = train_rent_processed[~outliers_mask_rent].reset_index(drop=True)
print(f"去除异常值后 - 训练集: {train_rent_processed.shape}")



第二部分：特征工程

2.1 房价特征工程
特征工程后 - 训练集: (103871, 82), 测试集: (34017, 82)
房价异常值数量: 2027 (1.95%)
去除异常值后 - 训练集: (101844, 82)

2.2 租金特征工程
特征工程后 - 训练集: (98899, 77), 测试集: (9773, 77)
租金异常值数量: 1734 (1.75%)
去除异常值后 - 训练集: (97165, 77)


In [8]:

# 第三部分：模型构建
print("\n第三部分：模型构建")

# 3.1 房价模型训练
print("\n3.1 房价模型训练")

X_price = train_price_processed.drop('Price', axis=1)
y_price = train_price_processed['Price']
y_price_log = np.log(y_price)

if 'ID' in test_price_processed.columns:
    test_price_ids = test_price_processed['ID']
    X_test_price = test_price_processed.drop('ID', axis=1)
else:
    X_test_price = test_price_processed.copy()

numeric_features_price = X_price.select_dtypes(include=[np.number]).columns.tolist()
categorical_features_price = X_price.select_dtypes(include=['object']).columns.tolist()

scaler_price = StandardScaler()
X_price[numeric_features_price] = scaler_price.fit_transform(X_price[numeric_features_price])
X_test_price[numeric_features_price] = scaler_price.transform(X_test_price[numeric_features_price])

label_encoders_price = {}
for col in categorical_features_price:
    le = LabelEncoder()
    all_categories = pd.concat([X_price[col], X_test_price[col]]).unique()
    le.fit(all_categories)
    X_price[col] = le.transform(X_price[col])
    X_test_price[col] = le.transform(X_test_price[col])
    label_encoders_price[col] = le

X_train_price, X_val_price, y_train_price_log, y_val_price_log = train_test_split(
    X_price, y_price_log, test_size=0.2, random_state=111
)

lasso_selector_price = Lasso(alpha=0.001, random_state=111)
lasso_selector_price.fit(X_train_price, y_train_price_log)

feature_importance_price = pd.DataFrame({
    'feature': X_train_price.columns,
    'importance': np.abs(lasso_selector_price.coef_)
})
feature_importance_price = feature_importance_price.sort_values('importance', ascending=False)
selected_features_price = feature_importance_price.head(50)['feature'].tolist()

X_train_price_selected = X_train_price[selected_features_price]
X_val_price_selected = X_val_price[selected_features_price]
X_test_price_selected = X_test_price[selected_features_price]

print(f"特征筛选: {X_train_price.shape[1]} -> {len(selected_features_price)}")

def evaluate_model(model, X_train, y_train_log, X_val, y_val_log):
    model.fit(X_train, y_train_log)

    y_train_pred_log = model.predict(X_train)
    y_train_pred = np.exp(y_train_pred_log)
    y_train_actual = np.exp(y_train_log)
    mae_train = mean_absolute_error(y_train_actual, y_train_pred)
    rmse_train = np.sqrt(mean_squared_error(y_train_actual, y_train_pred))

    y_val_pred_log = model.predict(X_val)
    y_val_pred = np.exp(y_val_pred_log)
    y_val_actual = np.exp(y_val_log)
    mae_val = mean_absolute_error(y_val_actual, y_val_pred)
    rmse_val = np.sqrt(mean_squared_error(y_val_actual, y_val_pred))

    cv_scores_mae = []
    cv_scores_rmse = []
    kf = KFold(n_splits=6, shuffle=True, random_state=111)

    for train_idx, val_idx in kf.split(X_train):
        X_cv_train, X_cv_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_cv_train_log, y_cv_val_log = y_train_log.iloc[train_idx], y_train_log.iloc[val_idx]

        model.fit(X_cv_train, y_cv_train_log)
        y_cv_pred_log = model.predict(X_cv_val)
        y_cv_pred = np.exp(y_cv_pred_log)
        y_cv_actual = np.exp(y_cv_val_log)

        cv_scores_mae.append(mean_absolute_error(y_cv_actual, y_cv_pred))
        cv_scores_rmse.append(np.sqrt(mean_squared_error(y_cv_actual, y_cv_pred)))

    mae_cv = np.mean(cv_scores_mae)
    rmse_cv = np.mean(cv_scores_rmse)

    return {
        'MAE_train': mae_train,
        'RMSE_train': rmse_train,
        'MAE_val': mae_val,
        'RMSE_val': rmse_val,
        'MAE_cv': mae_cv,
        'RMSE_cv': rmse_cv
    }

print("\n训练房价模型...")

ols_model_price = LinearRegression()
ols_metrics_price = evaluate_model(ols_model_price, X_train_price_selected, y_train_price_log, X_val_price_selected, y_val_price_log)
print(f"OLS - MAE(样本内): {ols_metrics_price['MAE_train']:.2f}, MAE(样本外): {ols_metrics_price['MAE_val']:.2f}, MAE(CV): {ols_metrics_price['MAE_cv']:.2f}")

lasso_params = {'alpha': [  0.01, 0.1, 1.0, 10.0, 100.0]}
lasso_grid_price = GridSearchCV(Lasso(random_state=111, max_iter=10000), lasso_params, cv=6, scoring='neg_mean_absolute_error', n_jobs=-1)
lasso_grid_price.fit(X_train_price_selected, y_train_price_log)
lasso_best_price = lasso_grid_price.best_estimator_
lasso_metrics_price = evaluate_model(lasso_best_price, X_train_price_selected, y_train_price_log, X_val_price_selected, y_val_price_log)
print(f"Lasso - MAE(样本内): {lasso_metrics_price['MAE_train']:.2f}, MAE(样本外): {lasso_metrics_price['MAE_val']:.2f}, MAE(CV): {lasso_metrics_price['MAE_cv']:.2f}")

ridge_params = {'alpha': [ 0.01, 0.1, 1.0, 10.0, 100.0]}
ridge_grid_price = GridSearchCV(Ridge(random_state=111, max_iter=10000), ridge_params, cv=6, scoring='neg_mean_absolute_error', n_jobs=-1)
ridge_grid_price.fit(X_train_price_selected, y_train_price_log)
ridge_best_price = ridge_grid_price.best_estimator_
ridge_metrics_price = evaluate_model(ridge_best_price, X_train_price_selected, y_train_price_log, X_val_price_selected, y_val_price_log)
print(f"Ridge - MAE(样本内): {ridge_metrics_price['MAE_train']:.2f}, MAE(样本外): {ridge_metrics_price['MAE_val']:.2f}, MAE(CV): {ridge_metrics_price['MAE_cv']:.2f}")

elastic_params = {'alpha': [0.01, 0.1, 1.0, 10.0,100], 'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]}
elastic_grid_price = GridSearchCV(ElasticNet(random_state=111, max_iter=10000), elastic_params, cv=6, scoring='neg_mean_absolute_error', n_jobs=-1)
elastic_grid_price.fit(X_train_price_selected, y_train_price_log)
elastic_best_price = elastic_grid_price.best_estimator_
elastic_metrics_price = evaluate_model(elastic_best_price, X_train_price_selected, y_train_price_log, X_val_price_selected, y_val_price_log)
print(f"ElasticNet - MAE(样本内): {elastic_metrics_price['MAE_train']:.2f}, MAE(样本外): {elastic_metrics_price['MAE_val']:.2f}, MAE(CV): {elastic_metrics_price['MAE_cv']:.2f}")

results_price = pd.DataFrame({
    'Model': ['OLS', 'Lasso', 'Ridge', 'ElasticNet'],
    'MAE_In_Sample': [ols_metrics_price['MAE_train'], lasso_metrics_price['MAE_train'], ridge_metrics_price['MAE_train'], elastic_metrics_price['MAE_train']],
    'RMSE_In_Sample': [ols_metrics_price['RMSE_train'], lasso_metrics_price['RMSE_train'], ridge_metrics_price['RMSE_train'], elastic_metrics_price['RMSE_train']],
    'MAE_Out_Sample': [ols_metrics_price['MAE_val'], lasso_metrics_price['MAE_val'], ridge_metrics_price['MAE_val'], elastic_metrics_price['MAE_val']],
    'RMSE_Out_Sample': [ols_metrics_price['RMSE_val'], lasso_metrics_price['RMSE_val'], ridge_metrics_price['RMSE_val'], elastic_metrics_price['RMSE_val']],
    'MAE_CV': [ols_metrics_price['MAE_cv'], lasso_metrics_price['MAE_cv'], ridge_metrics_price['MAE_cv'], elastic_metrics_price['MAE_cv']],
    'RMSE_CV': [ols_metrics_price['RMSE_cv'], lasso_metrics_price['RMSE_cv'], ridge_metrics_price['RMSE_cv'], elastic_metrics_price['RMSE_cv']]
})
print("\n房价模型性能汇总:")
print(results_price.to_string(index=False))

# 3.2 租金模型训练
print("\n3.2 租金模型训练")

X_rent = train_rent_processed.drop('Price', axis=1)
y_rent = train_rent_processed['Price']
y_rent_log = np.log(y_rent)

if 'ID' in test_rent_processed.columns:
    test_rent_ids = test_rent_processed['ID']
    X_test_rent = test_rent_processed.drop('ID', axis=1)
else:
    X_test_rent = test_rent_processed.copy()

numeric_features_rent = X_rent.select_dtypes(include=[np.number]).columns.tolist()
categorical_features_rent = X_rent.select_dtypes(include=['object']).columns.tolist()

scaler_rent = StandardScaler()
X_rent[numeric_features_rent] = scaler_rent.fit_transform(X_rent[numeric_features_rent])
X_test_rent[numeric_features_rent] = scaler_rent.transform(X_test_rent[numeric_features_rent])

label_encoders_rent = {}
for col in categorical_features_rent:
    le = LabelEncoder()
    all_categories = pd.concat([X_rent[col], X_test_rent[col]]).unique()
    le.fit(all_categories)
    X_rent[col] = le.transform(X_rent[col])
    X_test_rent[col] = le.transform(X_test_rent[col])
    label_encoders_rent[col] = le

X_train_rent, X_val_rent, y_train_rent_log, y_val_rent_log = train_test_split(
    X_rent, y_rent_log, test_size=0.2, random_state=111
)

lasso_selector_rent = Lasso(alpha=0.001, random_state=111)
lasso_selector_rent.fit(X_train_rent, y_train_rent_log)

feature_importance_rent = pd.DataFrame({
    'feature': X_train_rent.columns,
    'importance': np.abs(lasso_selector_rent.coef_)
})
feature_importance_rent = feature_importance_rent.sort_values('importance', ascending=False)
selected_features_rent = feature_importance_rent.head(60)['feature'].tolist()

X_train_rent_selected = X_train_rent[selected_features_rent]
X_val_rent_selected = X_val_rent[selected_features_rent]
X_test_rent_selected = X_test_rent[selected_features_rent]

print(f"特征筛选: {X_train_rent.shape[1]} -> {len(selected_features_rent)}")

print("\n训练租金模型...")

ols_model_rent = LinearRegression()
ols_metrics_rent = evaluate_model(ols_model_rent, X_train_rent_selected, y_train_rent_log, X_val_rent_selected, y_val_rent_log)
print(f"OLS - MAE(样本内): {ols_metrics_rent['MAE_train']:.2f}, MAE(样本外): {ols_metrics_rent['MAE_val']:.2f}, MAE(CV): {ols_metrics_rent['MAE_cv']:.2f}")

lasso_grid_rent = GridSearchCV(Lasso(random_state=111, max_iter=10000), lasso_params, cv=6, scoring='neg_mean_absolute_error', n_jobs=-1)
lasso_grid_rent.fit(X_train_rent_selected, y_train_rent_log)
lasso_best_rent = lasso_grid_rent.best_estimator_
lasso_metrics_rent = evaluate_model(lasso_best_rent, X_train_rent_selected, y_train_rent_log, X_val_rent_selected, y_val_rent_log)
print(f"Lasso - MAE(样本内): {lasso_metrics_rent['MAE_train']:.2f}, MAE(样本外): {lasso_metrics_rent['MAE_val']:.2f}, MAE(CV): {lasso_metrics_rent['MAE_cv']:.2f}")

ridge_grid_rent = GridSearchCV(Ridge(random_state=111, max_iter=10000), ridge_params, cv=6, scoring='neg_mean_absolute_error', n_jobs=-1)
ridge_grid_rent.fit(X_train_rent_selected, y_train_rent_log)
ridge_best_rent = ridge_grid_rent.best_estimator_
ridge_metrics_rent = evaluate_model(ridge_best_rent, X_train_rent_selected, y_train_rent_log, X_val_rent_selected, y_val_rent_log)
print(f"Ridge - MAE(样本内): {ridge_metrics_rent['MAE_train']:.2f}, MAE(样本外): {ridge_metrics_rent['MAE_val']:.2f}, MAE(CV): {ridge_metrics_rent['MAE_cv']:.2f}")

elastic_grid_rent = GridSearchCV(ElasticNet(random_state=111, max_iter=10000), elastic_params, cv=6, scoring='neg_mean_absolute_error', n_jobs=-1)
elastic_grid_rent.fit(X_train_rent_selected, y_train_rent_log)
elastic_best_rent = elastic_grid_rent.best_estimator_
elastic_metrics_rent = evaluate_model(elastic_best_rent, X_train_rent_selected, y_train_rent_log, X_val_rent_selected, y_val_rent_log)
print(f"ElasticNet - MAE(样本内): {elastic_metrics_rent['MAE_train']:.2f}, MAE(样本外): {elastic_metrics_rent['MAE_val']:.2f}, MAE(CV): {elastic_metrics_rent['MAE_cv']:.2f}")

results_rent = pd.DataFrame({
    'Model': ['OLS', 'Lasso', 'Ridge', 'ElasticNet'],
    'MAE_In_Sample': [ols_metrics_rent['MAE_train'], lasso_metrics_rent['MAE_train'], ridge_metrics_rent['MAE_train'], elastic_metrics_rent['MAE_train']],
    'RMSE_In_Sample': [ols_metrics_rent['RMSE_train'], lasso_metrics_rent['RMSE_train'], ridge_metrics_rent['RMSE_train'], elastic_metrics_rent['RMSE_train']],
    'MAE_Out_Sample': [ols_metrics_rent['MAE_val'], lasso_metrics_rent['MAE_val'], ridge_metrics_rent['MAE_val'], elastic_metrics_rent['MAE_val']],
    'RMSE_Out_Sample': [ols_metrics_rent['RMSE_val'], lasso_metrics_rent['RMSE_val'], ridge_metrics_rent['RMSE_val'], elastic_metrics_rent['RMSE_val']],
    'MAE_CV': [ols_metrics_rent['MAE_cv'], lasso_metrics_rent['MAE_cv'], ridge_metrics_rent['MAE_cv'], elastic_metrics_rent['MAE_cv']],
    'RMSE_CV': [ols_metrics_rent['RMSE_cv'], lasso_metrics_rent['RMSE_cv'], ridge_metrics_rent['RMSE_cv'], elastic_metrics_rent['RMSE_cv']]
})
print("\n租金模型性能汇总:")
print(results_rent.to_string(index=False))


第三部分：模型构建

3.1 房价模型训练
特征筛选: 81 -> 50

训练房价模型...
OLS - MAE(样本内): 646466.28, MAE(样本外): 638831.46, MAE(CV): 647063.58
Lasso - MAE(样本内): 659344.56, MAE(样本外): 651103.63, MAE(CV): 659784.40
Ridge - MAE(样本内): 646466.28, MAE(样本外): 638831.47, MAE(CV): 647063.58


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


ElasticNet - MAE(样本内): 646938.57, MAE(样本外): 639703.79, MAE(CV): 647563.37

房价模型性能汇总:
     Model  MAE_In_Sample  RMSE_In_Sample  MAE_Out_Sample  RMSE_Out_Sample        MAE_CV      RMSE_CV
       OLS  646466.279208    1.081084e+06   638831.460098     1.084221e+06 647063.583300 1.084265e+06
     Lasso  659344.561965    1.095554e+06   651103.633064     1.097443e+06 659784.404248 1.101245e+06
     Ridge  646466.277164    1.081084e+06   638831.473525     1.084222e+06 647063.581667 1.084265e+06
ElasticNet  646938.571671    1.089729e+06   639703.787445     1.103863e+06 647563.371553 1.095661e+06

3.2 租金模型训练
特征筛选: 76 -> 60

训练租金模型...
OLS - MAE(样本内): 157992.35, MAE(样本外): 155788.04, MAE(CV): 158157.12


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Lasso - MAE(样本内): 161200.73, MAE(样本外): 158618.82, MAE(CV): 161269.53


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Ridge - MAE(样本内): 157992.35, MAE(样本外): 155788.04, MAE(CV): 158157.13


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


ElasticNet - MAE(样本内): 158246.97, MAE(样本外): 155830.82, MAE(CV): 158388.48

租金模型性能汇总:
     Model  MAE_In_Sample  RMSE_In_Sample  MAE_Out_Sample  RMSE_Out_Sample        MAE_CV       RMSE_CV
       OLS  157992.345242   252002.384746   155788.041937    249479.958664 158157.124337 252790.696607
     Lasso  161200.734592   256857.533881   158618.822258    253431.991171 161269.526095 256973.570411
     Ridge  157992.349381   252002.412256   155788.042724    249479.984987 158157.128671 252790.727660
ElasticNet  158246.974514   253102.959075   155830.818372    250170.994724 158388.484324 253695.956616


In [9]:

# 第四部分：生成提交文件
print("\n第五部分：生成提交文件")

models_dict_final = {
    'OLS': LinearRegression(),
    'Lasso': Lasso(alpha=lasso_grid_price.best_params_['alpha'], random_state=111, max_iter=10000),
    'Ridge': Ridge(alpha=ridge_grid_price.best_params_['alpha'], random_state=111, max_iter=10000),
    'ElasticNet': ElasticNet(alpha=elastic_grid_price.best_params_['alpha'], l1_ratio=elastic_grid_price.best_params_['l1_ratio'], random_state=111, max_iter=10000)
}

models_dict_rent_final = {
    'OLS': LinearRegression(),
    'Lasso': Lasso(alpha=lasso_grid_rent.best_params_['alpha'], random_state=111, max_iter=10000),
    'Ridge': Ridge(alpha=ridge_grid_rent.best_params_['alpha'], random_state=111, max_iter=10000),
    'ElasticNet': ElasticNet(alpha=elastic_grid_rent.best_params_['alpha'], l1_ratio=elastic_grid_rent.best_params_['l1_ratio'], random_state=111, max_iter=10000)
}

for model_name in ['OLS', 'Lasso', 'Ridge', 'ElasticNet']:
    print(f"\n生成 {model_name} 模型提交文件...")

    model_price = models_dict_final[model_name]
    model_price.fit(X_price[selected_features_price], y_price_log)
    y_test_pred_price_log = model_price.predict(X_test_price_selected)
    y_test_pred_price = np.exp(y_test_pred_price_log)

    model_rent = models_dict_rent_final[model_name]
    model_rent.fit(X_rent[selected_features_rent], y_rent_log)
    y_test_pred_rent_log = model_rent.predict(X_test_rent_selected)
    y_test_pred_rent = np.exp(y_test_pred_rent_log)

    submission_price = pd.DataFrame({
        'ID': test_price_ids if 'ID' in test_price_processed.columns else range(len(y_test_pred_price)),
        'Price': y_test_pred_price
    })

    submission_rent = pd.DataFrame({
        'ID': test_rent_ids if 'ID' in test_rent_processed.columns else range(len(y_test_pred_rent)),
        'Price': y_test_pred_rent
    })

    submission_combined = pd.concat([submission_price, submission_rent], ignore_index=True)

    filename = f'submission_{model_name}.csv'
    submission_combined.to_csv(filename, index=False)
    print(f"已保存: {filename} (样本数: {len(submission_combined)})")



第五部分：生成提交文件

生成 OLS 模型提交文件...
已保存: submission_OLS.csv (样本数: 43790)

生成 Lasso 模型提交文件...
已保存: submission_Lasso.csv (样本数: 43790)

生成 Ridge 模型提交文件...
已保存: submission_Ridge.csv (样本数: 43790)

生成 ElasticNet 模型提交文件...
已保存: submission_ElasticNet.csv (样本数: 43790)
